# Batch Photometry Demo

This notebook demonstrates the `spxquery.batch` module for extracting multi-source aperture photometry from SPHEREx full-frame images.

**Workflow:**
1. Prepare a source catalog (CSV with `targetid`, `ra`, `dec`)
2. Query IRSA TAP for full-frame images covering a sky region
3. Download the images
4. Extract photometry for all sources in each image's FOV
5. Aggregate into per-source light curves
6. Inspect results

**Key parameters for narrowing the search:**
- `bands`: Filter by detector band (D1–D6)
- `mjd_range`: Filter by time window `(mjd_min, mjd_max)`
- `coverage_mode`: `"full"` = region fully inside image; `"any"` = partial overlap

In [ ]:
import sys
from pathlib import Path

# Ensure spxquery from this repo is importable
sys.path.insert(0, str(Path("../../src").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spxquery.batch import BatchConfig, BatchPipeline
from spxquery.batch.config import load_catalog
from spxquery.core.config import PhotometryConfig, Source

## 1. Prepare a Source Catalog

We create a small catalog of known sources around the NEP (North Ecliptic Pole) region. In practice, you would load an existing catalog from DESI, SDSS, or your own observations.

In [ ]:
# Demo directory and output
DEMO_DIR = Path(".").resolve()
OUTPUT_DIR = DEMO_DIR / "batch_output"

# Create a small catalog of sources near the NEP
# These are real coordinates from the DESI NEP field
catalog_data = {
    "targetid": [
        39633451355209872,
        39633446326241333,
        39633456308686512,
        39633453871796114,
        39633461169884285,
        39633465875891054,
        39633443780300215,
        39633470535763603,
    ],
    "ra": [266.10, 268.40, 269.90, 270.50, 271.20, 271.80, 272.50, 273.30],
    "dec": [66.20, 66.50, 66.80, 66.30, 67.00, 66.60, 66.90, 67.30],
}

catalog_df = pd.DataFrame(catalog_data)
catalog_path = DEMO_DIR / "demo_catalog.csv"
catalog_df.to_csv(catalog_path, index=False)

print(f"Catalog: {len(catalog_df)} sources")
print(f"  RA range:  {catalog_df['ra'].min():.2f} - {catalog_df['ra'].max():.2f}")
print(f"  Dec range: {catalog_df['dec'].min():.2f} - {catalog_df['dec'].max():.2f}")
catalog_df

## 2. Configure and Run the Pipeline

`BatchConfig` defines the sky region, catalog, and processing parameters.

To keep this demo manageable (~18 images, ~1.3 GB), we filter by:
- **Band**: D3 only (1.63–2.41 $\mu$m)
- **MJD range**: A 2-day window
- **Coverage**: `"full"` — only images that fully contain the search circle

For production use, set `bands=None` and `mjd_range=None` to process all epochs and bands.

In [ ]:
config = BatchConfig(
    center_ra=270.0,
    center_dec=66.6,
    radius=0.3,                        # Degrees — small for demo
    catalog_path=catalog_path,
    output_dir=OUTPUT_DIR,
    coverage_mode="full",              # Region fully inside image
    bands=["D3"],                      # Single band for demo
    mjd_range=(60791.0, 60793.0),      # 2-day window (~18 images)
    max_images=50,                     # Safety cap
    max_download_workers=4,
    max_extract_workers=4,
    photometry=PhotometryConfig(
        aperture_method="fwhm",
        fwhm_multiplier=2.5,
        background_method="window",
        window_size=30,
        subtract_zodi=True,
    ),
)

print(f"Region: RA={config.center_ra}, Dec={config.center_dec}, radius={config.radius} deg")
print(f"Bands: {config.bands}")
print(f"MJD range: {config.mjd_range}")
print(f"Coverage: {config.coverage_mode}")
print(f"Output: {config.output_dir}")

In [ ]:
pipeline = BatchPipeline(config)

### Stage 1: Query

In [ ]:
query_results = pipeline.run_query()

print(f"\nFound {len(query_results)} observations")
print(f"Estimated download: ~{len(query_results) * 0.07:.1f} GB")
print("Bands:")
for band, count in sorted(query_results.band_counts.items()):
    print(f"  {band}: {count} images")

### Stage 2: Download

Full-frame images (~71.6 MB each).

In [ ]:
download_results = pipeline.run_download(skip_existing=True)

n_success = sum(1 for r in download_results if r.success)
print(f"\nDownloaded {n_success}/{len(download_results)} images")

### Stage 3: Extract Photometry

For each image, the pipeline:
1. Reads the MEF file once
2. Projects all catalog sources to pixel coordinates via WCS
3. Filters to sources within the field of view
4. Extracts aperture photometry for each in-FOV source
5. Saves results as a per-image CSV (incremental / resumable)

In [ ]:
n_new = pipeline.run_extract(skip_existing=True)
print(f"\nExtracted {n_new} per-image CSVs")

### Stage 4: Aggregate Light Curves

Combines per-image CSVs into per-source light curves using a memory-efficient bucket-based approach.

In [ ]:
n_sources = pipeline.run_aggregate(clean=True)
print(f"\nCreated {n_sources} light curve files")

## 3. Inspect Results

In [ ]:
# List all light curve files
lc_files = sorted(config.lightcurve_dir.glob("*.csv"))
print(f"Light curves: {len(lc_files)} files")

# Show directory structure
print(f"\nOutput structure:")
print(f"  {config.output_dir}/")
print(f"    images/          — {len(list(config.image_dir.rglob('*.fits')))} FITS files")
print(f"    per_image/       — {len(list(config.per_image_dir.glob('*.csv')))} CSV files")
print(f"    lightcurves/     — {len(lc_files)} CSV files")

In [ ]:
# Load and display a sample light curve
if lc_files:
    sample = lc_files[0]
    lc = pd.read_csv(sample)
    print(f"Sample: {sample.name} — {len(lc)} observations")
    display(lc.head(10))

In [ ]:
# Spectral view: flux vs wavelength for sources with multiple observations
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
axes = axes.ravel()

plotted = 0
for lc_file in lc_files:
    lc = pd.read_csv(lc_file)
    if len(lc) < 2:
        continue
    if plotted >= 4:
        break

    ax = axes[plotted]
    target_id = lc_file.stem

    ax.errorbar(
        lc["wavelength"],
        lc["flux"],
        yerr=lc["flux_error"],
        fmt="o",
        markersize=3,
        capsize=2,
        color="tab:blue",
    )

    ax.set_title(f"{target_id}", fontsize=9)
    ax.set_xlabel(r"$\lambda$ [$\mu$m]")
    if plotted % 2 == 0:
        ax.set_ylabel(r"Flux [$\mu$Jy]")
    ax.set_yscale("symlog", linthresh=10)
    plotted += 1

# Hide unused axes
for i in range(plotted, 4):
    axes[i].set_visible(False)

fig.suptitle("SPHEREx Batch Photometry — Sample Light Curves (D3 band)", fontsize=12)
plt.show()

## 4. Alternative: One-Line Execution

For production use, you can run the entire pipeline with a single function call:

In [ ]:
from spxquery.batch import run_batch

# This runs all four stages: query -> download -> extract -> aggregate
# Uncomment to execute:

# pipeline = run_batch(
#     catalog="demo_catalog.csv",
#     center_ra=270.0,
#     center_dec=66.6,
#     radius=0.3,
#     output_dir="batch_output_v2",
#     coverage_mode="full",
#     bands=["D3"],
#     mjd_range=(60791.0, 60793.0),
#     max_images=50,
#     max_extract_workers=4,
# )

## Notes

- **Incremental processing:** Stages 2 and 3 are resumable. Re-running with `skip_existing=True` skips already-processed files.
- **Size gate:** `max_images` prevents accidental large downloads. Increase it if you know what you're doing.
- **Coverage modes:** `"full"` ensures the search circle is entirely inside each image (recommended for photometry). `"any"` includes images with partial overlap.
- **Time filtering:** Use `mjd_range=(mjd_min, mjd_max)` to select specific epochs. `None` = all time.
- **Band filtering:** Use `bands=["D1", "D3"]` to select specific detectors. `None` = all bands.
- **Memory:** Aggregation uses a bucket-based approach that never loads the full dataset into memory.